In [1]:
%pip install librosa soundfile numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install webrtcvad-wheels
%pip install resemblyzer --no-deps
%pip install torch
%pip install typing

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# =========================================================
# SPEAKER VERIFICATION MODULE — Function form (integration-ready)
# =========================================================

import os
import numpy as np
from resemblyzer import VoiceEncoder, preprocess_wav

# Load model once (not inside function, so it's not reloaded every call)
_encoder = VoiceEncoder()


def build_reference_embedding(reference_audio_path: str) -> np.ndarray:
    """
    Builds and returns a voice embedding from a reference (registered) audio file.
    Call this once per registered speaker, save the returned embedding,
    and pass it into verify_speaker() for future checks.
    """
    if not os.path.exists(reference_audio_path):
        raise FileNotFoundError(f"Reference audio not found: {reference_audio_path}")

    ref_wav = preprocess_wav(reference_audio_path)
    embedding = _encoder.embed_utterance(ref_wav)
    return embedding


def verify_speaker(
    audio_path: str,
    reference_embedding: np.ndarray,
    claimed_identity: str = "User",
    threshold: float = 0.75
) -> dict:
    """
    Compares an incoming audio file against a given reference embedding.

    Args:
        audio_path: path to the audio file being verified
        reference_embedding: embedding returned by build_reference_embedding()
        claimed_identity: label of the identity being claimed (e.g. "CEO")
        threshold: similarity cutoff for a match (default 0.75)

    Returns:
        dict with speaker_match_score, speaker_verified, claimed_identity
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    test_wav = preprocess_wav(audio_path)
    test_embedding = _encoder.embed_utterance(test_wav)

    similarity = np.dot(reference_embedding, test_embedding) / (
        np.linalg.norm(reference_embedding) * np.linalg.norm(test_embedding)
    )
    similarity = float(similarity)

    return {
        "speaker_match_score": round(similarity, 2),
        "speaker_verified": similarity >= threshold,
        "claimed_identity": claimed_identity
    }

Loaded the voice encoder model on cpu in 0.11 seconds.


In [4]:
reference_embedding = build_reference_embedding(r"C:\Users\Simran\Downloads\audio_files\reference_audio_clean.wav")

result = verify_speaker(
    audio_path=r"C:\Users\Simran\Downloads\audio_files\test_audio_clean.wav",
    reference_embedding=reference_embedding,
    claimed_identity="CEO"
)

print(f"speaker_match_score: {result['speaker_match_score']}")
print(f"speaker_verified: {result['speaker_verified']}")
print(f"claimed_identity: {result['claimed_identity']}")

speaker_match_score: 0.64
speaker_verified: False
claimed_identity: CEO
